In [2]:
# Install causal-conv1d first, as it is a strict dependency for Mamba
!pip install causal-conv1d>=1.4.0 --no-build-isolation

# Install the official mamba-ssm package
!pip install mamba-ssm --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 5.0 MB/s eta 0:00:0000:01
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 46.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.7/338.7 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 MB 19.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 64.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 25.4 MB/s eta 0:00:00
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.2.post1-cp312-cp312-linux_x86_64.whl size=322288410 sha256=7a67070c1e7e99c95abd1319623f044e8a1b3fb46f774bfdea949f0a4fc79638
  Stored in directory: /root/.cache/pip/wheels/da/67/03/99148d6eeaa4ec2855d71295ac83bcbc8ba7b41a298

In [3]:
# ==============================================================================
# LYT-Mamba training script with SSM state dimension (d_state) ablation
# ------------------------------------------------------------------------------
# Reproduces a "TABLE IX: SSM state dimension sensitivity" style study.
# For each N in ABLATION_DSTATES a fresh model is trained, and the script logs:
#   N | best PSNR (dB) | best SSIM | Params (M) | FLOPs (G) | FPS | peak Mem (GB)
# plus peak training memory, appended to /kaggle/working/dstate_ablation.csv.
# The CSV is append-mode, so one N per Kaggle session also works: set
# ABLATION_DSTATES = [8] in session 1, [16] in session 2, and so on.
#
# Benchmark resolution (FLOPs / FPS / Mem) is 256x256, matching PATCH_SIZE.
# ==============================================================================

# ==============================================================================
# 1. Imports & Configuration
# ==============================================================================
import os
import csv
import sys
import random
import time
import subprocess
import math
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim

from mamba_ssm import Mamba
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'thop'], check=False)

# --- CONFIGURATION ---
# UPDATE THIS PATH to match your Kaggle dataset directory!
BASE_DIR   = Path('/kaggle/input/datasets/zhangyubin1/lol-v2/LOLv2/Synthetic')
TRAIN_LOW  = BASE_DIR / 'Train' / 'Low'
TRAIN_HIGH = BASE_DIR / 'Train' / 'Normal'
TEST_LOW   = BASE_DIR / 'Test' / 'Low'
TEST_HIGH  = BASE_DIR / 'Test' / 'Normal'

EPOCHS      = 2
BATCH_SIZE  = 2
LR          = 2e-4
PATCH_SIZE  = 256
FILTERS     = 32
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SAVE_DIR    = Path('/kaggle/working/checkpoints')
SAVE_DIR.mkdir(exist_ok=True, parents=True)

# --- SSM STATE DIMENSION ABLATION CONFIGURATION ---
RUN_ABLATION      = True                 # False -> single full training run at D_STATE
D_STATE           = 16                   # used only when RUN_ABLATION is False
ABLATION_DSTATES  = [8, 16, 32, 64]      # the N column of the sensitivity table
ABLATION_EPOCHS   = 2                    # epochs per ablation run; set 500 for final numbers
BENCH_INPUT       = (1, 3, 256, 256)     # (B, C, H, W); 256x256 benchmark resolution
CSV_PATH          = Path('/kaggle/working/dstate_ablation.csv')
SEED              = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ==============================================================================
# 2. Dataset Definitions
# ==============================================================================
class LOLDataset(Dataset):
    def __init__(self, low_dir, high_dir, patch_size=128, augment=True):
        self.low_paths  = sorted(list(Path(low_dir).glob('*.*')))
        self.high_paths = sorted(list(Path(high_dir).glob('*.*')))
        self.patch_size = patch_size
        self.augment    = augment

    def __len__(self): return len(self.low_paths)

    def __getitem__(self, idx):
        low  = Image.open(self.low_paths[idx]).convert('RGB')
        high = Image.open(self.high_paths[idx]).convert('RGB')

        i, j, h, w = T.RandomCrop.get_params(low, (self.patch_size, self.patch_size))
        low  = TF.crop(low,  i, j, h, w)
        high = TF.crop(high, i, j, h, w)

        if self.augment:
            if random.random() > 0.5:
                low, high = TF.hflip(low), TF.hflip(high)
            if random.random() > 0.5:
                low, high = TF.vflip(low), TF.vflip(high)
            k = random.choice([0, 1, 2, 3])
            low, high = TF.rotate(low, k * 90), TF.rotate(high, k * 90)

        return TF.to_tensor(low), TF.to_tensor(high)

class LOLTestDataset(Dataset):
    def __init__(self, low_dir, high_dir):
        self.low_paths  = sorted(list(Path(low_dir).glob('*.*')))
        self.high_paths = sorted(list(Path(high_dir).glob('*.*')))

    def __len__(self): return len(self.low_paths)

    def __getitem__(self, idx):
        low  = TF.to_tensor(Image.open(self.low_paths[idx]).convert('RGB'))
        high = TF.to_tensor(Image.open(self.high_paths[idx]).convert('RGB'))
        return low, high

# ==============================================================================
# 3. Model Architecture (LYT-Net + Mamba + CBAM + YDetailHead)
#    d_state is now threaded through every Mamba instantiation.
# ==============================================================================
class MambaBottleneck(nn.Module):
    def __init__(self, channels, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.mamba = Mamba(d_model=channels, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x):
        B, C, H, W = x.shape
        x_seq = x.flatten(2).transpose(1, 2)
        x_seq = self.norm(x_seq)
        out_seq = self.mamba(x_seq)
        out = out_seq.transpose(1, 2).reshape(B, C, H, W)
        return x + out

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 1)), nn.ReLU(True),
            nn.Linear(max(channels // reduction, 1), channels), nn.Sigmoid()
        )
    def forward(self, x):
        B, C, _, _ = x.shape
        w = self.pool(x).view(B, C)
        w = self.fc(w).view(B, C, 1, 1)
        return x * w

class MSEFBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.dw3  = nn.Conv2d(channels, channels, 3, 1, 1,  groups=channels)
        self.dw5  = nn.Conv2d(channels, channels, 5, 1, 2,  groups=channels)
        self.pw   = nn.Conv2d(channels * 2, channels, 1)
        self.se   = SEBlock(channels)
        self.act  = nn.ReLU(True)
    def forward(self, x):
        out = self.act(self.pw(torch.cat([self.dw3(x), self.dw5(x)], dim=1)))
        return x + self.se(out)

class CBAM(nn.Module):
    def __init__(self, channels, reduction=8, spatial_kernel=7):
        super().__init__()
        mid = max(channels // reduction, 1)
        self.mlp = nn.Sequential(nn.Flatten(), nn.Linear(channels, mid), nn.ReLU(True), nn.Linear(mid, channels))
        self.conv = nn.Conv2d(2, 1, spatial_kernel, 1, spatial_kernel // 2, bias=False)
    def forward(self, x):
        avg_c, mx_c = self.mlp(F.adaptive_avg_pool2d(x, 1)), self.mlp(F.adaptive_max_pool2d(x, 1))
        x = x * torch.sigmoid(avg_c + mx_c).unsqueeze(-1).unsqueeze(-1)
        avg_s, mx_s = x.mean(dim=1, keepdim=True), x.max(dim=1, keepdim=True).values
        w = torch.sigmoid(self.conv(torch.cat([avg_s, mx_s], dim=1)))
        return x * w

class YDetailHead(nn.Module):
    _YW = torch.tensor([0.299, 0.587, 0.114]).reshape(1, 3, 1, 1)
    def __init__(self, feat_ch, mid_ch=32, delta_bound=0.04):
        super().__init__()
        self.delta_bound = delta_bound
        kernel = torch.tensor([[0., 1., 0.], [1.,-4., 1.], [0., 1., 0.]], dtype=torch.float32).reshape(1, 1, 3, 3)
        self.register_buffer('lap_kernel', kernel)
        self.convs = nn.Sequential(
            nn.Conv2d(feat_ch + 1, mid_ch, 3, 1, 1, bias=False), nn.ReLU(True),
            nn.Conv2d(mid_ch, mid_ch, 3, 1, 2, dilation=2, bias=False), nn.ReLU(True),
            nn.Conv2d(mid_ch, mid_ch, 3, 1, 1, groups=mid_ch, bias=False), nn.ReLU(True)
        )
        self.out = nn.Conv2d(mid_ch, 1, 1)

    def forward(self, feat, base):
        w = self._YW.to(base.device)
        y = (base * w).sum(dim=1, keepdim=True)
        edge = F.conv2d(y, self.lap_kernel, padding=1)
        x = self.convs(torch.cat([feat, edge], dim=1))
        delta_y = torch.tanh(self.out(x)) * self.delta_bound
        return (base + delta_y * w).clamp(0., 1.)

class Denoiser_Mamba(nn.Module):
    def __init__(self, in_ch, f, d_state=16):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(in_ch, f, 3, 1, 1), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv2d(f, f*2, 3, 2, 1), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv2d(f*2, f*4, 3, 2, 1), nn.ReLU(True))
        self.bottleneck = MambaBottleneck(channels=f*4, d_state=d_state)
        self.dec3 = nn.Sequential(nn.ConvTranspose2d(f*4, f*2, 2, 2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(f*4, f, 2, 2), nn.ReLU(True))
        self.dec1 = nn.Sequential(nn.Conv2d(f*2, f, 3, 1, 1), nn.ReLU(True))
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(e1); e3 = self.enc3(e2)
        d3 = self.dec3(self.bottleneck(e3))
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        return self.dec1(torch.cat([d2, e1], dim=1))

class LYTNet_Mamba(nn.Module):
    def __init__(self, filters=32, d_state=16):
        super().__init__()
        f = filters
        self.lum_pool  = nn.MaxPool2d(2)
        self.lum_proj  = nn.Conv2d(1, f, 1)
        self.lum_mamba = MambaBottleneck(channels=f, d_state=d_state)
        self.lum_up    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.lum_merge = nn.Conv2d(f * 2, f, 1)
        self.y_feat    = nn.Sequential(nn.Conv2d(1, f, 3, 1, 1), nn.ReLU(True))

        self.cb_denoiser = Denoiser_Mamba(1, f, d_state=d_state)
        self.cr_denoiser = Denoiser_Mamba(1, f, d_state=d_state)
        self.ref_conv    = nn.Conv2d(f * 2, f, 1)
        self.lum_conv    = nn.Conv2d(f, f, 1)
        self.msef        = MSEFBlock(f)
        self.cbam        = CBAM(f)

        self.recombine = nn.Sequential(nn.Conv2d(f * 2, f, 3, 1, 1), nn.ReLU(True))
        self.final_adj = nn.Sequential(nn.Conv2d(f, 3, 3, 1, 1), nn.Sigmoid())
        self.ydh       = YDetailHead(feat_ch=f)

    def forward(self, x):
        r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
        y  =  0.299*r + 0.587*g + 0.114*b
        cb = -0.16874*r - 0.33126*g + 0.5*b + 0.5
        cr =  0.5*r - 0.41869*g - 0.08131*b + 0.5

        y_full = self.y_feat(y)
        y_up   = self.lum_up(self.lum_mamba(self.lum_proj(self.lum_pool(y))))
        if y_up.shape != y_full.shape: y_up = F.interpolate(y_up, size=y_full.shape[2:])
        lum_feat = self.lum_merge(torch.cat([y_full, y_up], dim=1))

        ref = self.ref_conv(torch.cat([self.cb_denoiser(cb), self.cr_denoiser(cr)], dim=1))
        ref = self.cbam(self.msef(ref + 0.2 * self.lum_conv(lum_feat)))

        recomb = self.recombine(torch.cat([lum_feat, ref], dim=1))
        base = self.final_adj(recomb)
        return self.ydh(recomb, base), base

# ==============================================================================
# 4. Losses & Evaluation Helpers
# ==============================================================================
class HybridLoss(nn.Module):
    def __init__(self, lambda_ssim=0.05, lambda_edge=0.1, lambda_freq=0.1, lambda_aux=0.5):
        super().__init__()
        self.l1 = nn.L1Loss()
        self.l_ssim, self.l_edge, self.l_freq, self.l_aux = lambda_ssim, lambda_edge, lambda_freq, lambda_aux
        kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
        ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('kx', kx); self.register_buffer('ky', ky)

    def _gray(self, x): return 0.299*x[:,0:1] + 0.587*x[:,1:2] + 0.114*x[:,2:3]

    def forward(self, out, base, gt):
        l1 = self.l1(out, gt)
        aux = self.l1(base, gt)

        fft_p = torch.fft.fft2(self._gray(out))
        fft_g = torch.fft.fft2(self._gray(gt))
        freq = self.l1(fft_p.abs(), fft_g.abs())

        gp, gg = self._gray(out), self._gray(gt)
        mag_p = torch.sqrt(F.conv2d(gp, self.kx, padding=1)**2 + F.conv2d(gp, self.ky, padding=1)**2 + 1e-6)
        mag_g = torch.sqrt(F.conv2d(gg, self.kx, padding=1)**2 + F.conv2d(gg, self.ky, padding=1)**2 + 1e-6)
        edge = self.l1(mag_p, mag_g)

        return l1 + self.l_edge*edge + self.l_freq*freq + self.l_aux*aux

def gt_mean_gamma(pred_np, gt_np):
    mean_p, mean_g = pred_np.mean() + 1e-8, gt_np.mean() + 1e-8
    return np.clip(pred_np * (mean_g / mean_p), 0., 1.)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    psnr_list, ssim_list = [], []
    for low, high in loader:
        low, high = low.to(device), high.to(device)
        out, _ = model(low)
        for i in range(out.shape[0]):
            pred_np = out[i].cpu().numpy().transpose(1, 2, 0)
            gt_np   = high[i].cpu().numpy().transpose(1, 2, 0)
            pred_np = gt_mean_gamma(pred_np, gt_np)
            psnr_list.append(calc_psnr(gt_np, pred_np, data_range=1.0))
            ssim_list.append(calc_ssim(gt_np, pred_np, data_range=1.0, channel_axis=2))
    return np.mean(psnr_list), np.mean(ssim_list)

# ==============================================================================
# 5. Model Complexity, Speed & Memory Benchmarking
# ==============================================================================
def count_parameters(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def compute_flops_macs(model, input_size=(1, 3, 256, 256)):
    from thop import profile, clever_format

    device = next(model.parameters()).device
    dummy = torch.randn(*input_size, device=device)

    was_training = model.training
    model.eval()
    with torch.no_grad():
        macs, params = profile(model, inputs=(dummy,), verbose=False)
    if was_training:
        model.train()

    flops = macs * 2
    macs_f, flops_f, params_f = clever_format([macs, flops, params], "%.3f")
    return macs, flops, params, macs_f, flops_f, params_f


def mamba_block_flops(mamba, L, B=1):
    d_model = mamba.d_model
    d_state = mamba.d_state
    d_conv  = getattr(mamba, 'd_conv', 4)
    d_inner = getattr(mamba, 'd_inner', int(getattr(mamba, 'expand', 2) * d_model))
    dt_rank = getattr(mamba, 'dt_rank', math.ceil(d_model / 16))

    macs  = L * d_model * (2 * d_inner)
    macs += L * d_inner * d_conv
    macs += L * d_inner * (dt_rank + 2 * d_state)
    macs += L * dt_rank * d_inner
    macs += L * d_inner * d_model
    flops  = 2 * macs
    flops += 9 * L * d_inner * d_state
    flops += 2 * L * d_inner
    return B * flops


def compute_total_flops(model, input_size=BENCH_INPUT):
    from thop import profile

    device = next(model.parameters()).device
    dummy = torch.randn(*input_size, device=device)
    B = input_size[0]

    seq_lens, hooks = {}, []
    def make_hook(name):
        def hook(mod, inp, out):
            seq_lens[name] = inp[0].shape[1]
        return hook
    for name, mod in model.named_modules():
        if isinstance(mod, Mamba):
            hooks.append(mod.register_forward_hook(make_hook(name)))

    was_training = model.training
    model.eval()
    with torch.no_grad():
        _ = model(dummy)
    for h in hooks:
        h.remove()

    with torch.no_grad():
        macs, _ = profile(model, inputs=(dummy,), verbose=False)
    if was_training:
        model.train()
    conv_flops = macs * 2

    ssm_flops = 0
    for name, mod in model.named_modules():
        if isinstance(mod, Mamba):
            ssm_flops += mamba_block_flops(mod, seq_lens[name], B=B)

    return conv_flops + ssm_flops, conv_flops, ssm_flops


@torch.no_grad()
def measure_inference_speed(model, input_size=(1, 3, 256, 256), device=DEVICE,
                            warmup=20, runs=100):
    model.eval()
    dummy = torch.randn(*input_size, device=device)

    for _ in range(warmup):
        _ = model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()

    start = time.time()
    for _ in range(runs):
        _ = model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.time() - start

    batch = input_size[0]
    avg_time_ms = (elapsed / runs) * 1000.0
    fps = (runs * batch) / elapsed
    return avg_time_ms, fps


@torch.no_grad()
def measure_inference_memory(model, input_size=BENCH_INPUT, device=DEVICE):
    if device.type != 'cuda':
        return 0.0
    model.eval()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)
    dummy = torch.randn(*input_size, device=device)
    _ = model(dummy)
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated(device) / 1024**3


def benchmark_model(model, input_size=(1, 3, 256, 256), device=DEVICE,
                    warmup=20, runs=100):
    print("=" * 70)
    print(f"MODEL COMPLEXITY & SPEED BENCHMARK  (input size: {input_size})")
    print("=" * 70)

    total_params, trainable_params = count_parameters(model)
    print(f"Total Parameters:     {total_params:,}  ({total_params/1e6:.3f} M)")
    print(f"Trainable Parameters: {trainable_params:,}  ({trainable_params/1e6:.3f} M)")

    macs = flops = None
    try:
        total_flops, conv_flops, ssm_flops = compute_total_flops(model, input_size)
        flops = total_flops
        macs  = total_flops / 2
        print(f"FLOPs (conv/linear via thop): {conv_flops/1e9:.3f} G")
        print(f"FLOPs (SSM, analytical):      {ssm_flops/1e9:.3f} G")
        print(f"FLOPs (TOTAL):                {total_flops/1e9:.3f} G  (~{total_flops/2e9:.3f} GMACs)")
    except ImportError:
        print("FLOPs:                 [skipped] install with `pip install thop`")
    except Exception as e:
        print(f"FLOPs:                 [skipped] {e}")

    avg_time_ms, fps = measure_inference_speed(model, input_size, device, warmup, runs)
    peak_mem = measure_inference_memory(model, input_size, device)
    print(f"Avg Inference Time:    {avg_time_ms:.3f} ms/forward-pass  ({runs} runs, batch={input_size[0]})")
    print(f"Throughput:            {fps:.2f} FPS")
    print(f"Peak Inference Memory: {peak_mem:.2f} GB")
    print("=" * 70)

    return {
        'total_params': total_params,
        'trainable_params': trainable_params,
        'macs': macs,
        'flops': flops,
        'avg_inference_ms': avg_time_ms,
        'fps': fps,
        'peak_mem_GB': peak_mem,
    }

# ==============================================================================
# 6. Reusable Training Routine
# ==============================================================================
def train_model(model, train_loader, test_loader, epochs, tag=''):
    criterion = HybridLoss().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = torch.cuda.amp.GradScaler(enabled=True)

    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(DEVICE)

    best_psnr, best_ssim = 0.0, 0.0
    ckpt_path = SAVE_DIR / f'best_mamba{tag}.pth'

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        t0 = time.time()

        for low, high in train_loader:
            low, high = low.to(DEVICE), high.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=True):
                out, base = model(low)
                loss = criterion(out, base, high)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(train_loader)

        if epoch % 5 == 0 or epoch == epochs:
            psnr, ssim = evaluate(model, test_loader, DEVICE)
            elapsed = time.time() - t0
            print(f"[{tag or 'base'}] Epoch {epoch:3d}/{epochs} | Loss: {avg_loss:.4f} | "
                  f"PSNR: {psnr:.2f}dB | SSIM: {ssim:.4f} | {elapsed:.1f}s")

            if psnr > best_psnr:
                best_psnr, best_ssim = psnr, ssim
                torch.save(model.state_dict(), ckpt_path)
                print(f"  --> Saved new best! ({best_psnr:.2f}dB)")

    train_mem = torch.cuda.max_memory_allocated(DEVICE) / 1024**3 if DEVICE.type == 'cuda' else 0.0
    return best_psnr, best_ssim, train_mem

# ==============================================================================
# 7. SSM State Dimension Sensitivity Ablation (benchmarked @ 256x256)
# ==============================================================================
def append_csv_row(row):
    write_header = not CSV_PATH.exists()
    with open(CSV_PATH, 'a', newline='') as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(['d_state', 'epochs', 'best_psnr_dB', 'best_ssim',
                        'params_M', 'flops_G', 'fps', 'avg_ms',
                        'inf_mem_GB', 'train_mem_GB', 'bench_res'])
        w.writerow(row)


def print_sensitivity_table(results):
    print("\n" + "=" * 84)
    print(f"SSM STATE DIMENSION SENSITIVITY (running summary) @ {BENCH_INPUT[2]}x{BENCH_INPUT[3]}")
    print("=" * 84)
    print(f"{'N':>4} | {'PSNR (dB)':>9} | {'SSIM':>6} | {'Params':>8} | "
          f"{'FLOPs (G)':>9} | {'FPS':>6} | {'Mem (GB)':>8}")
    print("-" * 84)
    for r in results:
        print(f"{r['N']:>4} | {r['psnr']:>9.2f} | {r['ssim']:>6.4f} | "
              f"{r['params_M']:>7.3f}M | {r['flops_G']:>9.2f} | "
              f"{r['fps']:>6.1f} | {r['inf_mem']:>8.2f}")
    print("=" * 84)


def run_dstate_ablation(train_loader, test_loader):
    results = []
    for N in ABLATION_DSTATES:
        print("\n" + "#" * 72)
        print(f"# ABLATION RUN: d_state = {N}  ({ABLATION_EPOCHS} epochs)  "
              f"@ bench res {BENCH_INPUT[2]}x{BENCH_INPUT[3]}")
        print("#" * 72)

        set_seed(SEED)   # identical initialisation regime across runs
        model = LYTNet_Mamba(filters=FILTERS, d_state=N).to(DEVICE)
        total_params, _ = count_parameters(model)

        # FLOPs computed at the 256x256 benchmark resolution
        total_flops, conv_f, ssm_f = compute_total_flops(model, input_size=BENCH_INPUT)
        print(f"d_state={N} | Params: {total_params:,} ({total_params/1e6:.3f} M) | "
              f"FLOPs@{BENCH_INPUT[2]}x{BENCH_INPUT[3]}: {total_flops/1e9:.2f} G "
              f"(conv {conv_f/1e9:.2f} G + SSM {ssm_f/1e9:.2f} G)")

        t_start = time.time()
        best_psnr, best_ssim, train_mem = train_model(
            model, train_loader, test_loader, epochs=ABLATION_EPOCHS, tag=f'_N{N}')
        run_minutes = (time.time() - t_start) / 60.0

        # Speed & memory measured at the same 256x256 resolution
        avg_ms, fps = measure_inference_speed(model, input_size=BENCH_INPUT, device=DEVICE)
        inf_mem     = measure_inference_memory(model, input_size=BENCH_INPUT, device=DEVICE)

        print(f"d_state={N} DONE in {run_minutes:.1f} min | PSNR {best_psnr:.2f} dB | "
              f"SSIM {best_ssim:.4f} | {total_params/1e6:.3f} M | {fps:.1f} FPS | "
              f"inf mem {inf_mem:.2f} GB | train mem {train_mem:.2f} GB")

        results.append({'N': N, 'psnr': best_psnr, 'ssim': best_ssim,
                        'params_M': total_params / 1e6, 'flops_G': total_flops / 1e9,
                        'fps': fps, 'inf_mem': inf_mem})

        append_csv_row([N, ABLATION_EPOCHS, f"{best_psnr:.2f}", f"{best_ssim:.4f}",
                        f"{total_params/1e6:.3f}", f"{total_flops/1e9:.2f}",
                        f"{fps:.1f}", f"{avg_ms:.2f}",
                        f"{inf_mem:.2f}", f"{train_mem:.2f}",
                        f"{BENCH_INPUT[2]}x{BENCH_INPUT[3]}"])

        print_sensitivity_table(results)   # partial table survives an interrupted sweep

        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    return results

# ==============================================================================
# 8. Main Execution Block
# ==============================================================================
if __name__ == '__main__':
    print(f'Using Device: {DEVICE}')

    if not TRAIN_LOW.exists():
        print(f"ERROR: Dataset path {BASE_DIR} not found. Please add the LOL dataset to Kaggle.")
    else:
        set_seed(SEED)
        train_ds = LOLDataset(TRAIN_LOW, TRAIN_HIGH, patch_size=PATCH_SIZE)
        test_ds  = LOLTestDataset(TEST_LOW, TEST_HIGH)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
        test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)
        print(f"Data ready: {len(train_ds)} train | {len(test_ds)} test")

        if RUN_ABLATION:
            run_dstate_ablation(train_loader, test_loader)
            print(f"\nAblation complete. Results appended to {CSV_PATH}")
        else:
            model = LYTNet_Mamba(filters=FILTERS, d_state=D_STATE).to(DEVICE)
            benchmark_model(model, input_size=(1, 3, PATCH_SIZE, PATCH_SIZE), device=DEVICE)

            best_psnr, best_ssim, train_mem = train_model(
                model, train_loader, test_loader, epochs=EPOCHS, tag=f'_N{D_STATE}')

            print("\nFinal model benchmark (post-training):")
            benchmark_model(model, input_size=(1, 3, PATCH_SIZE, PATCH_SIZE), device=DEVICE)
            print(f"Training Complete. Best PSNR: {best_psnr:.2f}dB | Best SSIM: {best_ssim:.4f} | "
                  f"Peak train mem: {train_mem:.2f} GB")

Using Device: cuda
Data ready: 900 train | 100 test

########################################################################
# ABLATION RUN: d_state = 8  (2 epochs)  @ bench res 256x256
########################################################################
d_state=8 | Params: 598,286 (0.598 M) | FLOPs@256x256: 22.23 G (conv 19.99 G + SSM 2.25 G)


/tmp/ipykernel_57/1578159963.py:467: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler(enabled=True)


[_N8] Epoch   2/2 | Loss: 0.8862 | PSNR: 19.14dB | SSIM: 0.6852 | 29.6s
  --> Saved new best! (19.14dB)
d_state=8 DONE in 1.0 min | PSNR 19.14 dB | SSIM 0.6852 | 0.598 M | 73.5 FPS | inf mem 0.10 GB | train mem 0.39 GB

SSM STATE DIMENSION SENSITIVITY (running summary) @ 256x256
   N | PSNR (dB) |   SSIM |   Params | FLOPs (G) |    FPS | Mem (GB)
------------------------------------------------------------------------------------
   8 |     19.14 | 0.6852 |   0.598M |     22.23 |   73.5 |     0.10

########################################################################
# ABLATION RUN: d_state = 16  (2 epochs)  @ bench res 256x256
########################################################################
d_state=16 | Params: 612,110 (0.612 M) | FLOPs@256x256: 22.56 G (conv 19.99 G + SSM 2.57 G)
[_N16] Epoch   2/2 | Loss: 1.1642 | PSNR: 12.38dB | SSIM: 0.1535 | 32.3s
  --> Saved new best! (12.38dB)
d_state=16 DONE in 1.0 min | PSNR 12.38 dB | SSIM 0.1535 | 0.612 M | 65.7 FPS | inf mem 0.1

In [ ]:
# ==============================================================================
# LYT-Mamba training script with SSM state dimension (d_state) ablation
# ------------------------------------------------------------------------------
# Reproduces a "TABLE IX: SSM state dimension sensitivity" style study.
# For each N in ABLATION_DSTATES a fresh model is trained, and the script logs:
#   N | best PSNR (dB) | best SSIM | Params (M) | FLOPs (G) | FPS | peak Mem (GB)
# plus peak training memory, appended to /kaggle/working/dstate_ablation.csv.
# The CSV is append-mode, so one N per Kaggle session also works: set
# ABLATION_DSTATES = [8] in session 1, [16] in session 2, and so on.
#
# The only architectural knob varied is `d_state` of every Mamba block:
#   MambaBottleneck(d_state=N)   <- the N column of the table
# now threaded through Denoiser_Mamba and LYTNet_Mamba (previously the callers
# never passed it, so all runs silently used the default of 16).
# ==============================================================================
 
# ==============================================================================
# 1. Imports & Configuration
# ==============================================================================
import os
import csv
import sys
import random
import time
import subprocess
import math
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim
 
from mamba_ssm import Mamba
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'thop'], check=False)
 
# --- CONFIGURATION ---
# UPDATE THIS PATH to match your Kaggle dataset directory!
BASE_DIR   = Path('/kaggle/input/datasets/zhangyubin1/lol-v2/LOLv2/Synthetic')
TRAIN_LOW  = BASE_DIR / 'Train' / 'Low'
TRAIN_HIGH = BASE_DIR / 'Train' / 'Normal'
TEST_LOW   = BASE_DIR / 'Test' / 'Low'
TEST_HIGH  = BASE_DIR / 'Test' / 'Normal'
 
EPOCHS      = 2
BATCH_SIZE  = 2
LR          = 2e-4
PATCH_SIZE  = 256
FILTERS     = 32
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
 
SAVE_DIR    = Path('/kaggle/working/checkpoints')
SAVE_DIR.mkdir(exist_ok=True, parents=True)
 
# --- SSM STATE DIMENSION ABLATION CONFIGURATION ---
RUN_ABLATION      = True                 # False -> single full training run at D_STATE
D_STATE           = 16                   # used only when RUN_ABLATION is False
ABLATION_DSTATES  = [8, 16, 32, 64]      # the N column of the sensitivity table
ABLATION_EPOCHS   = 2                 # epochs per ablation run; set 500 for final numbers
BENCH_INPUT       = (1, 3, 400, 600)     # (B, C, H, W); LOL native 400 x 600 resolution
CSV_PATH          = Path('/kaggle/working/dstate_ablation.csv')
SEED              = 42
 
 
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
 
# ==============================================================================
# 2. Dataset Definitions
# ==============================================================================
class LOLDataset(Dataset):
    def __init__(self, low_dir, high_dir, patch_size=128, augment=True):
        self.low_paths  = sorted(list(Path(low_dir).glob('*.*')))
        self.high_paths = sorted(list(Path(high_dir).glob('*.*')))
        self.patch_size = patch_size
        self.augment    = augment
 
    def __len__(self): return len(self.low_paths)
 
    def __getitem__(self, idx):
        low  = Image.open(self.low_paths[idx]).convert('RGB')
        high = Image.open(self.high_paths[idx]).convert('RGB')
 
        i, j, h, w = T.RandomCrop.get_params(low, (self.patch_size, self.patch_size))
        low  = TF.crop(low,  i, j, h, w)
        high = TF.crop(high, i, j, h, w)
 
        if self.augment:
            if random.random() > 0.5:
                low, high = TF.hflip(low), TF.hflip(high)
            if random.random() > 0.5:
                low, high = TF.vflip(low), TF.vflip(high)
            k = random.choice([0, 1, 2, 3])
            low, high = TF.rotate(low, k * 90), TF.rotate(high, k * 90)
 
        return TF.to_tensor(low), TF.to_tensor(high)
 
class LOLTestDataset(Dataset):
    def __init__(self, low_dir, high_dir):
        self.low_paths  = sorted(list(Path(low_dir).glob('*.*')))
        self.high_paths = sorted(list(Path(high_dir).glob('*.*')))
 
    def __len__(self): return len(self.low_paths)
 
    def __getitem__(self, idx):
        low  = TF.to_tensor(Image.open(self.low_paths[idx]).convert('RGB'))
        high = TF.to_tensor(Image.open(self.high_paths[idx]).convert('RGB'))
        return low, high
 
# ==============================================================================
# 3. Model Architecture (LYT-Net + Mamba + CBAM + YDetailHead)
#    d_state is now threaded through every Mamba instantiation.
# ==============================================================================
class MambaBottleneck(nn.Module):
    def __init__(self, channels, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.mamba = Mamba(d_model=channels, d_state=d_state, d_conv=d_conv, expand=expand)
 
    def forward(self, x):
        B, C, H, W = x.shape
        x_seq = x.flatten(2).transpose(1, 2)
        x_seq = self.norm(x_seq)
        out_seq = self.mamba(x_seq)
        out = out_seq.transpose(1, 2).reshape(B, C, H, W)
        return x + out
 
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 1)), nn.ReLU(True),
            nn.Linear(max(channels // reduction, 1), channels), nn.Sigmoid()
        )
    def forward(self, x):
        B, C, _, _ = x.shape
        w = self.pool(x).view(B, C)
        w = self.fc(w).view(B, C, 1, 1)
        return x * w
 
class MSEFBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.dw3  = nn.Conv2d(channels, channels, 3, 1, 1,  groups=channels)
        self.dw5  = nn.Conv2d(channels, channels, 5, 1, 2,  groups=channels)
        self.pw   = nn.Conv2d(channels * 2, channels, 1)
        self.se   = SEBlock(channels)
        self.act  = nn.ReLU(True)
    def forward(self, x):
        out = self.act(self.pw(torch.cat([self.dw3(x), self.dw5(x)], dim=1)))
        return x + self.se(out)
 
class CBAM(nn.Module):
    def __init__(self, channels, reduction=8, spatial_kernel=7):
        super().__init__()
        mid = max(channels // reduction, 1)
        self.mlp = nn.Sequential(nn.Flatten(), nn.Linear(channels, mid), nn.ReLU(True), nn.Linear(mid, channels))
        self.conv = nn.Conv2d(2, 1, spatial_kernel, 1, spatial_kernel // 2, bias=False)
    def forward(self, x):
        avg_c, mx_c = self.mlp(F.adaptive_avg_pool2d(x, 1)), self.mlp(F.adaptive_max_pool2d(x, 1))
        x = x * torch.sigmoid(avg_c + mx_c).unsqueeze(-1).unsqueeze(-1)
        avg_s, mx_s = x.mean(dim=1, keepdim=True), x.max(dim=1, keepdim=True).values
        w = torch.sigmoid(self.conv(torch.cat([avg_s, mx_s], dim=1)))
        return x * w
 
class YDetailHead(nn.Module):
    _YW = torch.tensor([0.299, 0.587, 0.114]).reshape(1, 3, 1, 1)
    def __init__(self, feat_ch, mid_ch=32, delta_bound=0.04):
        super().__init__()
        self.delta_bound = delta_bound
        kernel = torch.tensor([[0., 1., 0.], [1.,-4., 1.], [0., 1., 0.]], dtype=torch.float32).reshape(1, 1, 3, 3)
        self.register_buffer('lap_kernel', kernel)
        self.convs = nn.Sequential(
            nn.Conv2d(feat_ch + 1, mid_ch, 3, 1, 1, bias=False), nn.ReLU(True),
            nn.Conv2d(mid_ch, mid_ch, 3, 1, 2, dilation=2, bias=False), nn.ReLU(True),
            nn.Conv2d(mid_ch, mid_ch, 3, 1, 1, groups=mid_ch, bias=False), nn.ReLU(True)
        )
        self.out = nn.Conv2d(mid_ch, 1, 1)
 
    def forward(self, feat, base):
        w = self._YW.to(base.device)
        y = (base * w).sum(dim=1, keepdim=True)
        edge = F.conv2d(y, self.lap_kernel, padding=1)
        x = self.convs(torch.cat([feat, edge], dim=1))
        delta_y = torch.tanh(self.out(x)) * self.delta_bound
        return (base + delta_y * w).clamp(0., 1.)
 
class Denoiser_Mamba(nn.Module):
    def __init__(self, in_ch, f, d_state=16):                       # <-- CHANGE 1
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(in_ch, f, 3, 1, 1), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv2d(f, f*2, 3, 2, 1), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv2d(f*2, f*4, 3, 2, 1), nn.ReLU(True))
        self.bottleneck = MambaBottleneck(channels=f*4, d_state=d_state)   # <-- CHANGE 1
        self.dec3 = nn.Sequential(nn.ConvTranspose2d(f*4, f*2, 2, 2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(f*4, f, 2, 2), nn.ReLU(True))
        self.dec1 = nn.Sequential(nn.Conv2d(f*2, f, 3, 1, 1), nn.ReLU(True))
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(e1); e3 = self.enc3(e2)
        d3 = self.dec3(self.bottleneck(e3))
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        return self.dec1(torch.cat([d2, e1], dim=1))
 
class LYTNet_Mamba(nn.Module):
    def __init__(self, filters=32, d_state=16):                     # <-- CHANGE 2
        super().__init__()
        f = filters
        self.lum_pool  = nn.MaxPool2d(2)
        self.lum_proj  = nn.Conv2d(1, f, 1)
        self.lum_mamba = MambaBottleneck(channels=f, d_state=d_state)      # <-- CHANGE 2
        self.lum_up    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.lum_merge = nn.Conv2d(f * 2, f, 1)
        self.y_feat    = nn.Sequential(nn.Conv2d(1, f, 3, 1, 1), nn.ReLU(True))
 
        self.cb_denoiser = Denoiser_Mamba(1, f, d_state=d_state)          # <-- CHANGE 2
        self.cr_denoiser = Denoiser_Mamba(1, f, d_state=d_state)          # <-- CHANGE 2
        self.ref_conv    = nn.Conv2d(f * 2, f, 1)
        self.lum_conv    = nn.Conv2d(f, f, 1)
        self.msef        = MSEFBlock(f)
        self.cbam        = CBAM(f)
 
        self.recombine = nn.Sequential(nn.Conv2d(f * 2, f, 3, 1, 1), nn.ReLU(True))
        self.final_adj = nn.Sequential(nn.Conv2d(f, 3, 3, 1, 1), nn.Sigmoid())
        self.ydh       = YDetailHead(feat_ch=f)
 
    def forward(self, x):
        r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
        y  =  0.299*r + 0.587*g + 0.114*b
        cb = -0.16874*r - 0.33126*g + 0.5*b + 0.5
        cr =  0.5*r - 0.41869*g - 0.08131*b + 0.5
 
        y_full = self.y_feat(y)
        y_up   = self.lum_up(self.lum_mamba(self.lum_proj(self.lum_pool(y))))
        if y_up.shape != y_full.shape: y_up = F.interpolate(y_up, size=y_full.shape[2:])
        lum_feat = self.lum_merge(torch.cat([y_full, y_up], dim=1))
 
        ref = self.ref_conv(torch.cat([self.cb_denoiser(cb), self.cr_denoiser(cr)], dim=1))
        ref = self.cbam(self.msef(ref + 0.2 * self.lum_conv(lum_feat)))
 
        recomb = self.recombine(torch.cat([lum_feat, ref], dim=1))
        base = self.final_adj(recomb)
        return self.ydh(recomb, base), base
 
# ==============================================================================
# 4. Losses & Evaluation Helpers
# ==============================================================================
class HybridLoss(nn.Module):
    def __init__(self, lambda_ssim=0.05, lambda_edge=0.1, lambda_freq=0.1, lambda_aux=0.5):
        super().__init__()
        self.l1 = nn.L1Loss()
        self.l_ssim, self.l_edge, self.l_freq, self.l_aux = lambda_ssim, lambda_edge, lambda_freq, lambda_aux
        kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
        ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('kx', kx); self.register_buffer('ky', ky)
 
    def _gray(self, x): return 0.299*x[:,0:1] + 0.587*x[:,1:2] + 0.114*x[:,2:3]
 
    def forward(self, out, base, gt):
        l1 = self.l1(out, gt)
        aux = self.l1(base, gt)
 
        # Frequency Loss (Perceptual Proxy)
        fft_p = torch.fft.fft2(self._gray(out))
        fft_g = torch.fft.fft2(self._gray(gt))
        freq = self.l1(fft_p.abs(), fft_g.abs())
 
        # Edge Loss
        gp, gg = self._gray(out), self._gray(gt)
        mag_p = torch.sqrt(F.conv2d(gp, self.kx, padding=1)**2 + F.conv2d(gp, self.ky, padding=1)**2 + 1e-6)
        mag_g = torch.sqrt(F.conv2d(gg, self.kx, padding=1)**2 + F.conv2d(gg, self.ky, padding=1)**2 + 1e-6)
        edge = self.l1(mag_p, mag_g)
 
        return l1 + self.l_edge*edge + self.l_freq*freq + self.l_aux*aux
 
def gt_mean_gamma(pred_np, gt_np):
    mean_p, mean_g = pred_np.mean() + 1e-8, gt_np.mean() + 1e-8
    return np.clip(pred_np * (mean_g / mean_p), 0., 1.)
 
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    psnr_list, ssim_list = [], []
    for low, high in loader:
        low, high = low.to(device), high.to(device)
        out, _ = model(low)
        for i in range(out.shape[0]):
            pred_np = out[i].cpu().numpy().transpose(1, 2, 0)
            gt_np   = high[i].cpu().numpy().transpose(1, 2, 0)
            pred_np = gt_mean_gamma(pred_np, gt_np)
            psnr_list.append(calc_psnr(gt_np, pred_np, data_range=1.0))
            ssim_list.append(calc_ssim(gt_np, pred_np, data_range=1.0, channel_axis=2))
    return np.mean(psnr_list), np.mean(ssim_list)
 
# ==============================================================================
# 5. Model Complexity, Speed & Memory Benchmarking
# ==============================================================================
def count_parameters(model):
    """Returns (total_params, trainable_params). Exact, includes Mamba weights."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable
 
 
def compute_flops_macs(model, input_size=(1, 3, 600, 400)):
    """
    MACs and FLOPs via `thop`. Convention: 1 MAC = 2 FLOPs.
    NOTE: thop cannot see inside the custom CUDA selective-scan kernel of
    mamba_ssm.Mamba, so the reported figures are a lower bound dominated by
    the convolutional parts. Parameter counts remain exact.
    """
    from thop import profile, clever_format
 
    device = next(model.parameters()).device
    dummy = torch.randn(*input_size, device=device)
 
    was_training = model.training
    model.eval()
    with torch.no_grad():
        macs, params = profile(model, inputs=(dummy,), verbose=False)
    if was_training:
        model.train()
 
    flops = macs * 2
    macs_f, flops_f, params_f = clever_format([macs, flops, params], "%.3f")
    return macs, flops, params, macs_f, flops_f, params_f
 
 
def mamba_block_flops(mamba, L, B=1):
    """
    Analytical FLOPs for one mamba_ssm.Mamba block on a length-L sequence.
    Needed because the fused CUDA fast path bypasses the nn.Linear/Conv1d
    module forwards, so hook-based profilers (thop) record zero for the
    entire block and the count would not vary with d_state. The selective
    scan is accounted as 9*L*D*N FLOPs following the convention used by
    the VMamba/Vim FLOPs counters; projections count 2 FLOPs per MAC.
    """
    d_model = mamba.d_model
    d_state = mamba.d_state
    d_conv  = getattr(mamba, 'd_conv', 4)
    d_inner = getattr(mamba, 'd_inner', int(getattr(mamba, 'expand', 2) * d_model))
    dt_rank = getattr(mamba, 'dt_rank', math.ceil(d_model / 16))
 
    macs  = L * d_model * (2 * d_inner)             # in_proj (x and z branches)
    macs += L * d_inner * d_conv                    # depthwise causal conv1d
    macs += L * d_inner * (dt_rank + 2 * d_state)   # x_proj -> delta, B, C
    macs += L * dt_rank * d_inner                   # dt_proj
    macs += L * d_inner * d_model                   # out_proj
    flops  = 2 * macs
    flops += 9 * L * d_inner * d_state              # selective scan recurrence
    flops += 2 * L * d_inner                        # D skip connection and z gating
    return B * flops
 
 
def compute_total_flops(model, input_size=BENCH_INPUT):
    """
    Total forward FLOPs = thop count of the standard conv/linear layers
                        + analytical FLOPs of every Mamba block.
    Returns (total_flops, conv_flops, ssm_flops).
    The sequence length seen by each Mamba block is captured with forward
    hooks, so the figure stays correct if the architecture or the input
    resolution changes. Assumes the CUDA fast path (standard on Kaggle
    GPUs), under which thop sees none of the Mamba internals.
    """
    from thop import profile
 
    device = next(model.parameters()).device
    dummy = torch.randn(*input_size, device=device)
    B = input_size[0]
 
    seq_lens, hooks = {}, []
    def make_hook(name):
        def hook(mod, inp, out):
            seq_lens[name] = inp[0].shape[1]        # input is (B, L, d_model)
        return hook
    for name, mod in model.named_modules():
        if isinstance(mod, Mamba):
            hooks.append(mod.register_forward_hook(make_hook(name)))
 
    was_training = model.training
    model.eval()
    with torch.no_grad():
        _ = model(dummy)                            # populate seq_lens
    for h in hooks:
        h.remove()
 
    with torch.no_grad():
        macs, _ = profile(model, inputs=(dummy,), verbose=False)
    if was_training:
        model.train()
    conv_flops = macs * 2                           # Mamba internals invisible to thop
 
    ssm_flops = 0
    for name, mod in model.named_modules():
        if isinstance(mod, Mamba):
            ssm_flops += mamba_block_flops(mod, seq_lens[name], B=B)
 
    return conv_flops + ssm_flops, conv_flops, ssm_flops
 
 
@torch.no_grad()
def measure_inference_speed(model, input_size=(1, 3, 600, 400), device=DEVICE,
                            warmup=20, runs=100):
    """Returns (avg_ms_per_image, fps) on a fixed-size dummy tensor."""
    model.eval()
    dummy = torch.randn(*input_size, device=device)
 
    for _ in range(warmup):
        _ = model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
 
    start = time.time()
    for _ in range(runs):
        _ = model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.time() - start
 
    batch = input_size[0]
    avg_time_ms = (elapsed / runs) * 1000.0
    fps = (runs * batch) / elapsed
    return avg_time_ms, fps
 
 
@torch.no_grad()
def measure_inference_memory(model, input_size=BENCH_INPUT, device=DEVICE):
    """Peak GPU memory (GB) for a single forward pass at input_size.
    This is the value to report in the Mem (GB) column of the table."""
    if device.type != 'cuda':
        return 0.0
    model.eval()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)
    dummy = torch.randn(*input_size, device=device)
    _ = model(dummy)
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated(device) / 1024**3
 
 
def benchmark_model(model, input_size=(1, 3, 600, 400), device=DEVICE,
                    warmup=20, runs=100):
    """Prints and returns params, MACs/FLOPs, inference time, FPS, peak memory."""
    print("=" * 70)
    print(f"MODEL COMPLEXITY & SPEED BENCHMARK  (input size: {input_size})")
    print("=" * 70)
 
    total_params, trainable_params = count_parameters(model)
    print(f"Total Parameters:     {total_params:,}  ({total_params/1e6:.3f} M)")
    print(f"Trainable Parameters: {trainable_params:,}  ({trainable_params/1e6:.3f} M)")
 
    macs = flops = None
    try:
        total_flops, conv_flops, ssm_flops = compute_total_flops(model, input_size)
        flops = total_flops
        macs  = total_flops / 2
        print(f"FLOPs (conv/linear via thop): {conv_flops/1e9:.3f} G")
        print(f"FLOPs (SSM, analytical):      {ssm_flops/1e9:.3f} G")
        print(f"FLOPs (TOTAL):                {total_flops/1e9:.3f} G  (~{total_flops/2e9:.3f} GMACs)")
    except ImportError:
        print("FLOPs:                 [skipped] install with `pip install thop`")
    except Exception as e:
        print(f"FLOPs:                 [skipped] {e}")
 
    avg_time_ms, fps = measure_inference_speed(model, input_size, device, warmup, runs)
    peak_mem = measure_inference_memory(model, input_size, device)
    print(f"Avg Inference Time:    {avg_time_ms:.3f} ms/forward-pass  ({runs} runs, batch={input_size[0]})")
    print(f"Throughput:            {fps:.2f} FPS")
    print(f"Peak Inference Memory: {peak_mem:.2f} GB")
    print("=" * 70)
 
    return {
        'total_params': total_params,
        'trainable_params': trainable_params,
        'macs': macs,
        'flops': flops,
        'avg_inference_ms': avg_time_ms,
        'fps': fps,
        'peak_mem_GB': peak_mem,
    }
 
# ==============================================================================
# 6. Reusable Training Routine
# ==============================================================================
def train_model(model, train_loader, test_loader, epochs, tag=''):
    """
    Trains one model instance from scratch.
    Returns (best_psnr, best_ssim, train_peak_mem_GB).
    The best checkpoint is saved to SAVE_DIR / f'best_mamba{tag}.pth'.
    """
    criterion = HybridLoss().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = torch.cuda.amp.GradScaler(enabled=True)
 
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(DEVICE)
 
    best_psnr, best_ssim = 0.0, 0.0
    ckpt_path = SAVE_DIR / f'best_mamba{tag}.pth'
 
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        t0 = time.time()
 
        for low, high in train_loader:
            low, high = low.to(DEVICE), high.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
 
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=True):
                out, base = model(low)
                loss = criterion(out, base, high)
 
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
 
            epoch_loss += loss.item()
 
        scheduler.step()
        avg_loss = epoch_loss / len(train_loader)
 
        if epoch % 5 == 0 or epoch == epochs:
            psnr, ssim = evaluate(model, test_loader, DEVICE)
            elapsed = time.time() - t0
            print(f"[{tag or 'base'}] Epoch {epoch:3d}/{epochs} | Loss: {avg_loss:.4f} | "
                  f"PSNR: {psnr:.2f}dB | SSIM: {ssim:.4f} | {elapsed:.1f}s")
 
            if psnr > best_psnr:
                best_psnr, best_ssim = psnr, ssim
                torch.save(model.state_dict(), ckpt_path)
                print(f"  --> Saved new best! ({best_psnr:.2f}dB)")
 
    train_mem = torch.cuda.max_memory_allocated(DEVICE) / 1024**3 if DEVICE.type == 'cuda' else 0.0
    return best_psnr, best_ssim, train_mem
 
# ==============================================================================
# 7. SSM State Dimension Sensitivity Ablation
# ==============================================================================
def append_csv_row(row):
    write_header = not CSV_PATH.exists()
    with open(CSV_PATH, 'a', newline='') as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(['d_state', 'epochs', 'best_psnr_dB', 'best_ssim',
                        'params_M', 'flops_G', 'fps', 'avg_ms',
                        'inf_mem_GB', 'train_mem_GB'])
        w.writerow(row)
 
 
def print_sensitivity_table(results):
    print("\n" + "=" * 84)
    print("SSM STATE DIMENSION SENSITIVITY (running summary)")
    print("=" * 84)
    print(f"{'N':>4} | {'PSNR (dB)':>9} | {'SSIM':>6} | {'Params':>8} | "
          f"{'FLOPs (G)':>9} | {'FPS':>6} | {'Mem (GB)':>8}")
    print("-" * 84)
    for r in results:
        print(f"{r['N']:>4} | {r['psnr']:>9.2f} | {r['ssim']:>6.4f} | "
              f"{r['params_M']:>7.3f}M | {r['flops_G']:>9.2f} | "
              f"{r['fps']:>6.1f} | {r['inf_mem']:>8.2f}")
    print("=" * 84)
 
 
def run_dstate_ablation(train_loader, test_loader):
    results = []
    for N in ABLATION_DSTATES:
        print("\n" + "#" * 72)
        print(f"# ABLATION RUN: d_state = {N}  ({ABLATION_EPOCHS} epochs)")
        print("#" * 72)
 
        set_seed(SEED)   # identical initialisation regime across runs
        model = LYTNet_Mamba(filters=FILTERS, d_state=N).to(DEVICE)
        total_params, _ = count_parameters(model)
        total_flops, conv_f, ssm_f = compute_total_flops(model, input_size=BENCH_INPUT)
        print(f"d_state={N} | Params: {total_params:,} ({total_params/1e6:.3f} M) | "
              f"FLOPs: {total_flops/1e9:.2f} G (conv {conv_f/1e9:.2f} G + SSM {ssm_f/1e9:.2f} G)")
 
        t_start = time.time()
        best_psnr, best_ssim, train_mem = train_model(
            model, train_loader, test_loader, epochs=ABLATION_EPOCHS, tag=f'_N{N}')
        run_minutes = (time.time() - t_start) / 60.0
 
        avg_ms, fps = measure_inference_speed(model, input_size=BENCH_INPUT, device=DEVICE)
        inf_mem     = measure_inference_memory(model, input_size=BENCH_INPUT, device=DEVICE)
 
        print(f"d_state={N} DONE in {run_minutes:.1f} min | PSNR {best_psnr:.2f} dB | "
              f"SSIM {best_ssim:.4f} | {total_params/1e6:.3f} M | {fps:.1f} FPS | "
              f"inf mem {inf_mem:.2f} GB | train mem {train_mem:.2f} GB")
 
        results.append({'N': N, 'psnr': best_psnr, 'ssim': best_ssim,
                        'params_M': total_params / 1e6, 'flops_G': total_flops / 1e9,
                        'fps': fps, 'inf_mem': inf_mem})
 
        append_csv_row([N, ABLATION_EPOCHS, f"{best_psnr:.2f}", f"{best_ssim:.4f}",
                        f"{total_params/1e6:.3f}", f"{total_flops/1e9:.2f}",
                        f"{fps:.1f}", f"{avg_ms:.2f}",
                        f"{inf_mem:.2f}", f"{train_mem:.2f}"])
 
        print_sensitivity_table(results)   # partial table survives an interrupted sweep
 
        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    return results
 
# ==============================================================================
# 8. Main Execution Block
# ==============================================================================
if __name__ == '__main__':
    print(f'Using Device: {DEVICE}')
 
    if not TRAIN_LOW.exists():
        print(f"ERROR: Dataset path {BASE_DIR} not found. Please add the LOL dataset to Kaggle.")
    else:
        set_seed(SEED)
        train_ds = LOLDataset(TRAIN_LOW, TRAIN_HIGH, patch_size=PATCH_SIZE)
        test_ds  = LOLTestDataset(TEST_LOW, TEST_HIGH)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
        test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)
        print(f"Data ready: {len(train_ds)} train | {len(test_ds)} test")
 
        if RUN_ABLATION:
            run_dstate_ablation(train_loader, test_loader)
            print(f"\nAblation complete. Results appended to {CSV_PATH}")
        else:
            model = LYTNet_Mamba(filters=FILTERS, d_state=D_STATE).to(DEVICE)
            benchmark_model(model, input_size=(1, 3, PATCH_SIZE, PATCH_SIZE), device=DEVICE)
 
            best_psnr, best_ssim, train_mem = train_model(
                model, train_loader, test_loader, epochs=EPOCHS, tag=f'_N{D_STATE}')
 
            print("\nFinal model benchmark (post-training):")
            benchmark_model(model, input_size=(1, 3, PATCH_SIZE, PATCH_SIZE), device=DEVICE)
            print(f"Training Complete. Best PSNR: {best_psnr:.2f}dB | Best SSIM: {best_ssim:.4f} | "
                  f"Peak train mem: {train_mem:.2f} GB")

Using Device: cuda
Data ready: 900 train | 100 test

########################################################################
# ABLATION RUN: d_state = 8  (2 epochs)
########################################################################
d_state=8 | Params: 598,286 (0.598 M) | FLOPs: 81.42 G (conv 73.20 G + SSM 8.23 G)


/tmp/ipykernel_57/2213142109.py:505: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler(enabled=True)


[_N8] Epoch   2/2 | Loss: 0.8940 | PSNR: 19.12dB | SSIM: 0.6938 | 31.1s
  --> Saved new best! (19.12dB)
d_state=8 DONE in 0.9 min | PSNR 19.12 dB | SSIM 0.6938 | 0.598 M | 18.6 FPS | inf mem 1.03 GB | train mem 1.10 GB

SSM STATE DIMENSION SENSITIVITY (running summary)
   N | PSNR (dB) |   SSIM |   Params | FLOPs (G) |    FPS | Mem (GB)
------------------------------------------------------------------------------------
   8 |     19.12 | 0.6938 |   0.598M |     81.42 |   18.6 |     1.03

########################################################################
# ABLATION RUN: d_state = 16  (2 epochs)
########################################################################
d_state=16 | Params: 612,110 (0.612 M) | FLOPs: 82.62 G (conv 73.20 G + SSM 9.42 G)
[_N16] Epoch   2/2 | Loss: 1.1636 | PSNR: 12.93dB | SSIM: 0.1839 | 32.1s
  --> Saved new best! (12.93dB)


In [5]:
# ==============================================================================
# 1. Imports & Configuration
# ==============================================================================
import os
import random
import time
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim

from mamba_ssm import Mamba

# --- CONFIGURATION ---
# UPDATE THIS PATH to match your Kaggle dataset directory!
BASE_DIR   = Path('/kaggle/input/datasets/zhangyubin1/lol-v2/LOLv2/Synthetic') 
TRAIN_LOW  = BASE_DIR / 'Train' / 'Low'
TRAIN_HIGH = BASE_DIR / 'Train' / 'Normal'
TEST_LOW   = BASE_DIR / 'Test' / 'Low'
TEST_HIGH  = BASE_DIR / 'Test' / 'Normal'

EPOCHS      = 500
BATCH_SIZE  = 2      # Kept at 4 to be safe with Mamba memory on 16GB T4
LR          = 2e-4
PATCH_SIZE  = 256
FILTERS     = 32
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SAVE_DIR    = Path('/kaggle/working/checkpoints')
SAVE_DIR.mkdir(exist_ok=True, parents=True)

# ==============================================================================
# 2. Dataset Definitions
# ==============================================================================
class LOLDataset(Dataset):
    def __init__(self, low_dir, high_dir, patch_size=128, augment=True):
        self.low_paths  = sorted(list(Path(low_dir).glob('*.*')))
        self.high_paths = sorted(list(Path(high_dir).glob('*.*')))
        self.patch_size = patch_size
        self.augment    = augment

    def __len__(self): return len(self.low_paths)

    def __getitem__(self, idx):
        low  = Image.open(self.low_paths[idx]).convert('RGB')
        high = Image.open(self.high_paths[idx]).convert('RGB')

        i, j, h, w = T.RandomCrop.get_params(low, (self.patch_size, self.patch_size))
        low  = TF.crop(low,  i, j, h, w)
        high = TF.crop(high, i, j, h, w)

        if self.augment:
            if random.random() > 0.5:
                low, high = TF.hflip(low), TF.hflip(high)
            if random.random() > 0.5:
                low, high = TF.vflip(low), TF.vflip(high)
            k = random.choice([0, 1, 2, 3])
            low, high = TF.rotate(low, k * 90), TF.rotate(high, k * 90)

        return TF.to_tensor(low), TF.to_tensor(high)

class LOLTestDataset(Dataset):
    def __init__(self, low_dir, high_dir):
        self.low_paths  = sorted(list(Path(low_dir).glob('*.*')))
        self.high_paths = sorted(list(Path(high_dir).glob('*.*')))

    def __len__(self): return len(self.low_paths)

    def __getitem__(self, idx):
        low  = TF.to_tensor(Image.open(self.low_paths[idx]).convert('RGB'))
        high = TF.to_tensor(Image.open(self.high_paths[idx]).convert('RGB'))
        return low, high

# ==============================================================================
# 3. Model Architecture (LYT-Net + Mamba + CBAM + YDetailHead)
# ==============================================================================
class MambaBottleneck(nn.Module):
    def __init__(self, channels, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.mamba = Mamba(d_model=channels, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x):
        B, C, H, W = x.shape
        x_seq = x.flatten(2).transpose(1, 2)
        x_seq = self.norm(x_seq)
        out_seq = self.mamba(x_seq)
        out = out_seq.transpose(1, 2).reshape(B, C, H, W)
        return x + out

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 1)), nn.ReLU(True),
            nn.Linear(max(channels // reduction, 1), channels), nn.Sigmoid()
        )
    def forward(self, x):
        B, C, _, _ = x.shape
        w = self.pool(x).view(B, C)
        w = self.fc(w).view(B, C, 1, 1)
        return x * w

class MSEFBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.dw3  = nn.Conv2d(channels, channels, 3, 1, 1,  groups=channels)
        self.dw5  = nn.Conv2d(channels, channels, 5, 1, 2,  groups=channels)
        self.pw   = nn.Conv2d(channels * 2, channels, 1)
        self.se   = SEBlock(channels)
        self.act  = nn.ReLU(True)
    def forward(self, x):
        out = self.act(self.pw(torch.cat([self.dw3(x), self.dw5(x)], dim=1)))
        return x + self.se(out)

class CBAM(nn.Module):
    def __init__(self, channels, reduction=8, spatial_kernel=7):
        super().__init__()
        mid = max(channels // reduction, 1)
        self.mlp = nn.Sequential(nn.Flatten(), nn.Linear(channels, mid), nn.ReLU(True), nn.Linear(mid, channels))
        self.conv = nn.Conv2d(2, 1, spatial_kernel, 1, spatial_kernel // 2, bias=False)
    def forward(self, x):
        avg_c, mx_c = self.mlp(F.adaptive_avg_pool2d(x, 1)), self.mlp(F.adaptive_max_pool2d(x, 1))
        x = x * torch.sigmoid(avg_c + mx_c).unsqueeze(-1).unsqueeze(-1)
        avg_s, mx_s = x.mean(dim=1, keepdim=True), x.max(dim=1, keepdim=True).values
        w = torch.sigmoid(self.conv(torch.cat([avg_s, mx_s], dim=1)))
        return x * w

class YDetailHead(nn.Module):
    _YW = torch.tensor([0.299, 0.587, 0.114]).reshape(1, 3, 1, 1)
    def __init__(self, feat_ch, mid_ch=32, delta_bound=0.04):
        super().__init__()
        self.delta_bound = delta_bound
        kernel = torch.tensor([[0., 1., 0.], [1.,-4., 1.], [0., 1., 0.]], dtype=torch.float32).reshape(1, 1, 3, 3)
        self.register_buffer('lap_kernel', kernel)
        self.convs = nn.Sequential(
            nn.Conv2d(feat_ch + 1, mid_ch, 3, 1, 1, bias=False), nn.ReLU(True),
            nn.Conv2d(mid_ch, mid_ch, 3, 1, 2, dilation=2, bias=False), nn.ReLU(True),
            nn.Conv2d(mid_ch, mid_ch, 3, 1, 1, groups=mid_ch, bias=False), nn.ReLU(True)
        )
        self.out = nn.Conv2d(mid_ch, 1, 1)

    def forward(self, feat, base):
        w = self._YW.to(base.device)
        y = (base * w).sum(dim=1, keepdim=True)
        edge = F.conv2d(y, self.lap_kernel, padding=1)
        x = self.convs(torch.cat([feat, edge], dim=1))
        delta_y = torch.tanh(self.out(x)) * self.delta_bound
        return (base + delta_y * w).clamp(0., 1.)

class Denoiser_Mamba(nn.Module):
    def __init__(self, in_ch, f):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(in_ch, f, 3, 1, 1), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv2d(f, f*2, 3, 2, 1), nn.ReLU(True))  
        self.enc3 = nn.Sequential(nn.Conv2d(f*2, f*4, 3, 2, 1), nn.ReLU(True)) 
        self.bottleneck = MambaBottleneck(channels=f*4)
        self.dec3 = nn.Sequential(nn.ConvTranspose2d(f*4, f*2, 2, 2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(f*4, f, 2, 2), nn.ReLU(True)) 
        self.dec1 = nn.Sequential(nn.Conv2d(f*2, f, 3, 1, 1), nn.ReLU(True))
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(e1); e3 = self.enc3(e2)
        d3 = self.dec3(self.bottleneck(e3))
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        return self.dec1(torch.cat([d2, e1], dim=1))

class LYTNet_Mamba(nn.Module):
    def __init__(self, filters=32):
        super().__init__()
        f = filters
        self.lum_pool  = nn.MaxPool2d(2)
        self.lum_proj  = nn.Conv2d(1, f, 1)    
        self.lum_mamba = MambaBottleneck(channels=f)
        self.lum_up    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.lum_merge = nn.Conv2d(f * 2, f, 1)  
        self.y_feat    = nn.Sequential(nn.Conv2d(1, f, 3, 1, 1), nn.ReLU(True))

        self.cb_denoiser = Denoiser_Mamba(1, f)
        self.cr_denoiser = Denoiser_Mamba(1, f)
        self.ref_conv    = nn.Conv2d(f * 2, f, 1)   
        self.lum_conv    = nn.Conv2d(f, f, 1)
        self.msef        = MSEFBlock(f)
        self.cbam        = CBAM(f)      

        self.recombine = nn.Sequential(nn.Conv2d(f * 2, f, 3, 1, 1), nn.ReLU(True))
        self.final_adj = nn.Sequential(nn.Conv2d(f, 3, 3, 1, 1), nn.Sigmoid())
        self.ydh       = YDetailHead(feat_ch=f) 

    def forward(self, x):
        r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
        y  =  0.299*r + 0.587*g + 0.114*b
        cb = -0.16874*r - 0.33126*g + 0.5*b + 0.5
        cr =  0.5*r - 0.41869*g - 0.08131*b + 0.5

        y_full = self.y_feat(y)                          
        y_up   = self.lum_up(self.lum_mamba(self.lum_proj(self.lum_pool(y))))
        if y_up.shape != y_full.shape: y_up = F.interpolate(y_up, size=y_full.shape[2:])
        lum_feat = self.lum_merge(torch.cat([y_full, y_up], dim=1))  

        ref = self.ref_conv(torch.cat([self.cb_denoiser(cb), self.cr_denoiser(cr)], dim=1))  
        ref = self.cbam(self.msef(ref + 0.2 * self.lum_conv(lum_feat)))

        recomb = self.recombine(torch.cat([lum_feat, ref], dim=1))  
        base = self.final_adj(recomb)   
        return self.ydh(recomb, base), base

# ==============================================================================
# 4. Losses & Evaluation Helpers
# ==============================================================================
class HybridLoss(nn.Module):
    def __init__(self, lambda_ssim=0.05, lambda_edge=0.1, lambda_freq=0.1, lambda_aux=0.5):
        super().__init__()
        self.l1 = nn.L1Loss()
        self.l_ssim, self.l_edge, self.l_freq, self.l_aux = lambda_ssim, lambda_edge, lambda_freq, lambda_aux
        kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
        ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('kx', kx); self.register_buffer('ky', ky)

    def _gray(self, x): return 0.299*x[:,0:1] + 0.587*x[:,1:2] + 0.114*x[:,2:3]

    def forward(self, out, base, gt):
        l1 = self.l1(out, gt)
        aux = self.l1(base, gt)
        
        # Frequency Loss (Perceptual Proxy)
        fft_p = torch.fft.fft2(self._gray(out))
        fft_g = torch.fft.fft2(self._gray(gt))
        freq = self.l1(fft_p.abs(), fft_g.abs())

        # Edge Loss
        gp, gg = self._gray(out), self._gray(gt)
        mag_p = torch.sqrt(F.conv2d(gp, self.kx, padding=1)**2 + F.conv2d(gp, self.ky, padding=1)**2 + 1e-6)
        mag_g = torch.sqrt(F.conv2d(gg, self.kx, padding=1)**2 + F.conv2d(gg, self.ky, padding=1)**2 + 1e-6)
        edge = self.l1(mag_p, mag_g)

        return l1 + self.l_edge*edge + self.l_freq*freq + self.l_aux*aux

def gt_mean_gamma(pred_np, gt_np):
    mean_p, mean_g = pred_np.mean() + 1e-8, gt_np.mean() + 1e-8
    return np.clip(pred_np * (mean_g / mean_p), 0., 1.)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    psnr_list, ssim_list = [], []
    for low, high in loader:
        low, high = low.to(device), high.to(device)
        out, _ = model(low)
        for i in range(out.shape[0]):
            pred_np = out[i].cpu().numpy().transpose(1, 2, 0)
            gt_np   = high[i].cpu().numpy().transpose(1, 2, 0)
            pred_np = gt_mean_gamma(pred_np, gt_np)
            psnr_list.append(calc_psnr(gt_np, pred_np, data_range=1.0))
            ssim_list.append(calc_ssim(gt_np, pred_np, data_range=1.0, channel_axis=2))
    return np.mean(psnr_list), np.mean(ssim_list)

# ==============================================================================
# 5. Main Training Execution Block
# ==============================================================================
if __name__ == '__main__':
    print(f'Using Device: {DEVICE}')
    
    # 1. Verify Dataset Paths
    if not TRAIN_LOW.exists():
        print(f"ERROR: Dataset path {BASE_DIR} not found. Please add the LOL dataset to Kaggle.")
    else:
        # 2. Init Loaders
        train_ds = LOLDataset(TRAIN_LOW, TRAIN_HIGH, patch_size=PATCH_SIZE)
        test_ds  = LOLTestDataset(TEST_LOW, TEST_HIGH)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
        test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)
        print(f"Data ready: {len(train_ds)} train | {len(test_ds)} test")

        # 3. Init Model & Optimizers
        model = LYTNet_Mamba(filters=FILTERS).to(DEVICE)
        criterion = HybridLoss().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
        scaler = torch.cuda.amp.GradScaler(enabled=True)

        # 4. Training Loop
        best_psnr = 0.0
        print("Starting Training...")
        for epoch in range(1, EPOCHS + 1):
            model.train()
            epoch_loss = 0.0
            t0 = time.time()

            for low, high in train_loader:
                low, high = low.to(DEVICE), high.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)

                # Use float16 for T4 compatibility
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=True):
                    out, base = model(low)
                    loss = criterion(out, base, high)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

                epoch_loss += loss.item()

            scheduler.step()
            avg_loss = epoch_loss / len(train_loader)

            # Evaluate every 5 epochs
            if epoch % 5 == 0 or epoch == EPOCHS:
                psnr, ssim = evaluate(model, test_loader, DEVICE)
                elapsed = time.time() - t0
                print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | PSNR: {psnr:.2f}dB | SSIM: {ssim:.4f} | {elapsed:.1f}s")

                if psnr > best_psnr:
                    best_psnr = psnr
                    torch.save(model.state_dict(), SAVE_DIR / 'best_mamba.pth')
                    print(f"  --> Saved new best! ({best_psnr:.2f}dB)")

        print(f"Training Complete. Best PSNR: {best_psnr:.2f}dB")

Using Device: cuda
Data ready: 900 train | 100 test
Starting Training...


/tmp/ipykernel_57/625864407.py:289: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)


KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# Mamba-LYTNet: Test, Evaluate, and Save Results
# ==============================================================================
import os
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim

# --- 1. Configuration ---
# Ensure these paths match your Kaggle environment
BASE_DIR   = Path('/kaggle/input/datasets/zhangyubin1/lol-v2/LOLv2/Synthetic') 
TRAIN_LOW  = BASE_DIR / 'Train' / 'Low'
TRAIN_HIGH = BASE_DIR / 'Train' / 'Normal'
TEST_LOW   = BASE_DIR / 'Test' / 'Low'
TEST_HIGH  = BASE_DIR / 'Test' / 'Normal'
WEIGHTS_PATH  = Path('/kaggle/working/checkpoints/best_mamba.pth')
RESULTS_DIR   = Path('/kaggle/working/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- 2. Evaluation Helper ---
def gt_mean_gamma(pred_np, gt_np):
    """Aligns mean brightness to ground truth to prevent unfair PSNR penalties."""
    mean_p = pred_np.mean() + 1e-8
    mean_g = gt_np.mean() + 1e-8
    return np.clip(pred_np * (mean_g / mean_p), 0., 1.)



In [ ]:


# --- 3. Main Execution ---
if __name__ == '__main__':
    # Initialize the model and load weights
    # (Assuming LYTNet_Mamba is already defined in your notebook from the previous cell)
    model = LYTNet_Mamba(filters=32).to(DEVICE)
    
    if not WEIGHTS_PATH.exists():
        raise FileNotFoundError(f"Weights not found at {WEIGHTS_PATH}. Please train the model first.")
        
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
    model.eval()
    print(f"✅ Loaded weights from {WEIGHTS_PATH}")

    # Prepare data paths
    low_files  = sorted(list(TEST_LOW.glob('*.*')))
    high_files = sorted(list(TEST_HIGH.glob('*.*')))
    
    psnr_list, ssim_list = [], []
    
    print("\nStarting Evaluation & Saving Results...\n" + "-"*50)
    
    # Optional: Setup a plot to show the first 3 results in the notebook
    n_show = min(3, len(low_files))
    fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
    if n_show == 1: axes = [axes] # Handle dimension if only 1 image

    with torch.no_grad():
        for idx, (low_path, high_path) in enumerate(zip(low_files, high_files)):
            # Load images
            low_img = Image.open(low_path).convert('RGB')
            high_img = Image.open(high_path).convert('RGB')
            
            low_tensor = TF.to_tensor(low_img).unsqueeze(0).to(DEVICE)
            gt_np = np.array(high_img).astype(np.float32) / 255.0
            
            # 1. Network Inference
            # The model returns (refined_output, base_output)
            out_tensor, _ = model(low_tensor)
            
            # 2. Post-processing
            pred_np = out_tensor.squeeze(0).cpu().numpy().transpose(1, 2, 0)
            pred_np = np.clip(pred_np, 0., 1.)
            
            # Apply GT-mean gamma correction
            pred_corrected = gt_mean_gamma(pred_np, gt_np)
            
            # 3. Metrics Calculation
            psnr = calc_psnr(gt_np, pred_corrected, data_range=1.0)
            ssim = calc_ssim(gt_np, pred_corrected, data_range=1.0, channel_axis=2)
            
            psnr_list.append(psnr)
            ssim_list.append(ssim)
            
            print(f"Image {idx+1:02d}: {low_path.name:15s} | PSNR: {psnr:.2f} dB | SSIM: {ssim:.4f}")
            
            # 4. Save the Enhanced Image to Disk
            save_path = RESULTS_DIR / f"enhanced_{low_path.name}"
            out_pil = Image.fromarray((pred_corrected * 255).astype(np.uint8))
            out_pil.save(save_path)

            # 5. Add to Plot (for the first few images)
            if idx < n_show:
                axes[idx][0].imshow(low_img)
                axes[idx][0].set_title("Input (Low Light)")
                axes[idx][0].axis('off')
                
                axes[idx][1].imshow(gt_np)
                axes[idx][1].set_title("Ground Truth")
                axes[idx][1].axis('off')
                
                axes[idx][2].imshow(pred_corrected)
                axes[idx][2].set_title(f"Mamba Enhanced\nPSNR: {psnr:.2f}dB")
                axes[idx][2].axis('off')

    # --- 4. Final Summary ---
    print("-" * 50)
    print(f"🏆 Final Average PSNR : {np.mean(psnr_list):.2f} dB")
    print(f"🏆 Final Average SSIM : {np.mean(ssim_list):.4f}")
    print(f"📁 All {len(low_files)} enhanced images saved to: {RESULTS_DIR}")
    
    plt.tight_layout()
    plt.show()